# Data Preparation

## 1. Select data

Data source: https://www.kaggle.com/datasets/sturarods/anac-national-civil-aviation-agency-2000-2025?select=anac_brazil.csv

Selected attriblutes with rationale:
- `nr_passag_pagos`: revenue passengers - primary target.
- `kg_carga_paga`: paid cargo - primary target.
- `nr_ano_mes_partida_real`: date in format YYYYMM for monthly aggregation.
- `ds_grupo_di`: filters for regular flights, excludes not regular (charters) and not productive (ferry/training).
- `ds_servico_tipo_linha`: filters cargo and passenger flights.
- `ds_natureza_tipo_linha`: separates domestic and international flights.
- `nm_pais_origem` and `nm_pais_destino`: restore missing values or not identified records in `ds_natureza_tipo_linha`.

The other attributes should be excluded because they are either derived from attributes mentioned above or do not carry useful information regarding national monthly passengers and cargo forecast.

## 2. Clean data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np

chunk_size = 100000
chunks = []

reader = pd.read_csv(
    'anac_brazil.csv',
    usecols=['ds_grupo_di', 'ds_natureza_tipo_linha', 'nm_pais_origem', 'nm_pais_destino', 'ds_servico_tipo_linha', 'nr_passag_pagos', 'nr_ano_mes_partida_real', 'kg_carga_paga'],
    chunksize=chunk_size
)

for chunk in reader:
    # filter regular flights
    chunk = chunk[chunk['ds_grupo_di'] == 'REGULAR']
    
    # identify domestic/international flights
    not_identified = chunk['ds_natureza_tipo_linha'] == 'NÃO IDENTIFICADO'
    international = (chunk['nm_pais_origem'] != 'BRASIL') | (chunk['nm_pais_destino'] != 'BRASIL')
    
    chunk['ds_natureza_tipo_linha'] = np.select(
        [
            not_identified & international, 
            not_identified & ~international
        ], 
        [
            'INTERNACIONAL', 
            'DOMÉSTICA'
        ], 
        default=chunk['ds_natureza_tipo_linha']
    )
    
    chunks.append(chunk)

In [3]:
df = pd.concat(chunks, ignore_index=True)

In [4]:
# drop nan dates, convert to date type
df = df.dropna(subset=['nr_ano_mes_partida_real'])
df['nr_ano_mes_partida_real'] = pd.to_datetime(df["nr_ano_mes_partida_real"], errors="coerce", format='%Y%m')

In [8]:
# filter valid passenger flights
passenger_flight = (df['nr_passag_pagos'] > 0.0) & (df['nr_passag_pagos'] <= 853.0) & (df['ds_servico_tipo_linha'].isin(['PASSAGEIRO', 'NÃO IDENTIFICADO']))
passengers = df[passenger_flight]

# passengers filed: convert to int type
passengers['nr_passag_pagos'] = passengers['nr_passag_pagos'].astype(int)

passengers = passengers[['nr_ano_mes_partida_real', 'ds_natureza_tipo_linha', 'nr_passag_pagos']]

In [9]:
# filter valid cargo flights
cargo_flight = (df['kg_carga_paga'] > 0.0) & (df['kg_carga_paga'] <= 140000.0)
cargo = df[cargo_flight]
cargo = cargo[['nr_ano_mes_partida_real', 'ds_natureza_tipo_linha', 'kg_carga_paga']]

In [11]:
# split each dataset into two datasets with domestic and international flights
domestic_pass = passengers[passengers['ds_natureza_tipo_linha'] == 'DOMÉSTICA']
international_pass = passengers[passengers['ds_natureza_tipo_linha'] == 'INTERNACIONAL']

domestic_cargo = cargo[cargo['ds_natureza_tipo_linha'] == 'DOMÉSTICA']
international_cargo = cargo[cargo['ds_natureza_tipo_linha'] == 'INTERNACIONAL']

# drop unnecessary column ds_natureza_tipo_linha from datasets
domestic_pass = domestic_pass.drop('ds_natureza_tipo_linha', axis=1)
international_pass = international_pass.drop('ds_natureza_tipo_linha', axis=1)

domestic_cargo = domestic_cargo.drop('ds_natureza_tipo_linha', axis=1)
international_cargo = international_cargo.drop('ds_natureza_tipo_linha', axis=1)

## 3. Construct data

### Derived attributes

- sliding window: capture short‑term momentum and seasonality.
- expanding window: capture long‑term trend information.


In [12]:
# group passengers by months and sort

domestic_pass_monthly = (
    domestic_pass
    .groupby('nr_ano_mes_partida_real')['nr_passag_pagos']
    .sum()
    .reset_index()
    .rename(columns={'nr_passag_pagos': 'passengers'})
)

international_pass_monthly = (
    international_pass
    .groupby('nr_ano_mes_partida_real')['nr_passag_pagos']
    .sum()
    .reset_index()
    .rename(columns={'nr_passag_pagos': 'passengers'})
)

domestic_pass_monthly = domestic_pass_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)
international_pass_monthly = international_pass_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)

In [13]:
# group cargo by months and sort

domestic_cargo_monthly = (
    domestic_cargo
    .groupby('nr_ano_mes_partida_real')['kg_carga_paga']
    .sum()
    .reset_index()
    .rename(columns={'kg_carga_paga': 'cargo'})
)

international_cargo_monthly = (
    international_cargo
    .groupby('nr_ano_mes_partida_real')['kg_carga_paga']
    .sum()
    .reset_index()
    .rename(columns={'kg_carga_paga': 'cargo'})
)

domestic_cargo_monthly = domestic_cargo_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)
international_cargo_monthly = international_cargo_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)

In [15]:
def make_derived_attributes(df, target):
    # sliding window
    df['lag_1']  = df[target].shift(1)
    df['lag_3']  = df[target].shift(3)
    df['lag_6']  = df[target].shift(6)
    df['lag_12'] = df[target].shift(12)

    # rolling means
    df['roll_mean_3'] = df[target].shift(1).rolling(window=3).mean()
    df['roll_mean_6'] = df[target].shift(1).rolling(window=6).mean()
    df['roll_mean_12'] = df[target].shift(1).rolling(window=12).mean()

    # rolling standard deviation
    df['roll_std_6'] = df[target].shift(1).rolling(window=6).std()

    # expanding window statistics
    df['exp_mean'] = df[target].shift(1).expanding().mean()
    df['exp_std']  = df[target].shift(1).expanding().std()
    df['exp_min']  = df[target].shift(1).expanding().min()
    df['exp_max']  = df[target].shift(1).expanding().max()

    return df

In [16]:
# add derived attributes to each dataset

domestic_pass_monthly = make_derived_attributes(domestic_pass_monthly, 'passengers')
international_pass_monthly = make_derived_attributes(international_pass_monthly, 'passengers')

domestic_cargo_monthly = make_derived_attributes(domestic_cargo_monthly, 'cargo')
international_cargo_monthly = make_derived_attributes(international_cargo_monthly, 'cargo')

## 4. Integrate data

The is no other data to merge with.

## 5. Format data

In [19]:
# translate column with date to English
domestic_pass_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)
international_pass_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)

domestic_cargo_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)
international_cargo_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)

In [20]:
# start datasets from 2008 to remove sparse data
domestic_pass_monthly = domestic_pass_monthly[domestic_pass_monthly['year_month'] >= '2008-01-01'].copy()
international_pass_monthly = international_pass_monthly[international_pass_monthly['year_month'] >= '2008-01-01'].copy()

domestic_cargo_monthly = domestic_cargo_monthly[domestic_cargo_monthly['year_month'] >= '2008-01-01'].copy()
international_cargo_monthly = international_cargo_monthly[international_cargo_monthly['year_month'] >= '2008-01-01'].copy()

Sample data:

In [21]:
domestic_pass_monthly.head()

,year_month,passengers,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,4111527,3853064.0,3261641.0,3955423.0,3989260.0,3.544392e+06,3.683011e+06,3.821791e+06,271795.471773,3.024862e+06,546774.864443,2109035.0,4275973.0
97,2008-02-01,4060498,4111527.0,3518470.0,3594543.0,3344750.0,3.827687e+06,3.709028e+06,3.831980e+06,308129.873611,3.036064e+06,554997.459942,2109035.0,4275973.0
98,2008-03-01,4396453,4060498.0,3853064.0,3914926.0,3738048.0,4.008363e+06,3.786688e+06,3.891625e+06,331348.245136,3.046518e+06,561743.279292,2109035.0,4275973.0
99,2008-04-01,4607122,4396453.0,4111527.0,3261641.0,4275973.0,4.189493e+06,3.866942e+06,3.946492e+06,416096.690867,3.060154e+06,575102.495478,2109035.0,4396453.0
100,2008-05-01,4933469,4607122.0,4060498.0,3518470.0,4266889.0,4.354691e+06,4.091189e+06,3.974088e+06,386118.922224,3.075623e+06,592733.630836,2109035.0,4607122.0


In [22]:
international_pass_monthly.head()

,year_month,passengers,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,468968,412477.0,350468.0,403574.0,339707.0,370535.000000,363952.833333,346335.833333,35404.430375,416184.562500,68592.490412,253080.0,589624.0
97,2008-02-01,443856,468968.0,348660.0,343351.0,296983.0,410035.000000,374851.833333,357107.583333,54795.962674,416728.721649,68444.449348,253080.0,589624.0
98,2008-03-01,473181,443856.0,412477.0,325187.0,331584.0,441767.000000,391602.666667,369347.000000,58478.573237,417005.530612,68145.846794,253080.0,589624.0
99,2008-04-01,425615,473181.0,468968.0,350468.0,348481.0,462001.666667,416268.333333,381146.750000,56022.014704,417572.959596,68031.946427,253080.0,589624.0
100,2008-05-01,437111,425615.0,443856.0,348660.0,329109.0,447550.666667,428792.833333,387574.583333,45845.017978,417653.380000,67692.255922,253080.0,589624.0


In [23]:
domestic_cargo_monthly.head()

,year_month,cargo,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,16261925.0,19182748.0,20453838.0,16161367.0,16255888.0,2.011575e+07,1.925906e+07,1.844076e+07,1.649099e+06,2.159548e+07,3.208429e+06,14073391.0,29708539.0
97,2008-02-01,18500543.0,16261925.0,20710678.0,19121448.0,15115814.0,1.871845e+07,1.927581e+07,1.844126e+07,1.611401e+06,2.154049e+07,3.237291e+06,14073391.0,29708539.0
98,2008-03-01,24112364.0,18500543.0,19182748.0,19924251.0,18263217.0,1.798174e+07,1.917233e+07,1.872332e+07,1.642926e+06,2.150947e+07,3.235168e+06,14073391.0,29708539.0
99,2008-04-01,24169469.0,24112364.0,16261925.0,20453838.0,18117040.0,1.962494e+07,1.987035e+07,1.921075e+07,2.623402e+06,2.153577e+07,3.229233e+06,14073391.0,29708539.0
100,2008-05-01,23648383.0,24169469.0,18500543.0,20710678.0,20023235.0,2.226079e+07,2.048962e+07,1.971512e+07,3.170242e+06,2.156210e+07,3.223659e+06,14073391.0,29708539.0


In [24]:
international_cargo_monthly.head()

,year_month,cargo,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,8879003.0,9083532.0,11753610.0,11192605.0,5912479.0,1.040571e+07,1.089176e+07,8.938208e+06,1.071708e+06,1.342957e+07,3.631049e+06,5912479.0,21342520.0
97,2008-02-01,9358577.0,8879003.0,10379982.0,12065484.0,6163845.0,9.447506e+06,1.050616e+07,9.185419e+06,1.327506e+06,1.338266e+07,3.641519e+06,5912479.0,21342520.0
98,2008-03-01,7011651.0,9358577.0,9083532.0,10875354.0,6172974.0,9.107037e+06,1.005501e+07,9.451646e+06,1.138033e+06,1.334160e+07,3.645434e+06,5912479.0,21342520.0
99,2008-04-01,7507680.0,7011651.0,8879003.0,11753610.0,7209370.0,8.416410e+06,9.411059e+06,9.521536e+06,1.585978e+06,1.327766e+07,3.682162e+06,5912479.0,21342520.0
100,2008-05-01,7722554.0,7507680.0,9358577.0,10379982.0,8264572.0,7.959303e+06,8.703404e+06,9.546395e+06,1.241554e+06,1.321996e+07,3.708678e+06,5912479.0,21342520.0


In [25]:
# save datasets to csv
international_pass_monthly.to_csv('international_pass_monthly.csv')
domestic_pass_monthly.to_csv('domestic_pass_monthly.csv')

international_cargo_monthly.to_csv('international_cargo_monthly.csv')
domestic_cargo_monthly.to_csv('domestic_cargo_monthly.csv')